# JiT-RCDM Training on Google Colab

Representation-Conditioned Diffusion Model — DinoV3 ViT-S/16 + JiT ViT + Flow Matching  
Dataset: Messidor-2 retinal fundus images (224×224)

## Before you start
1. **Runtime → Change runtime type → A100 GPU**
2. Upload the following to **Google Drive** under `MyDrive/jit_rcdm/`:
   - `train_reps.pt` — precomputed DinoV3 representations
   - `dinov3_vits16_tmp/` — DinoV3 checkpoint folder (`config.json` + `model.safetensors`)
   - `test_images/` — a few `.png` fundus images for sampling
3. Run cells **top to bottom**

## Workflow after local code changes
Edit locally → `git push` → re-run **Cell 3** in Colab → re-run **Cell 7** to reload imports.

## 1 — Check GPU

In [ ]:
import torch
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM    : {mem:.1f} GB")
    if mem >= 70:
        print("✓ A100/H100 80GB — use batch_size=256")
    elif mem >= 38:
        print("✓ A100 40GB — use batch_size=128")
    else:
        print("⚠ T4/V100 — use batch_size=32")
else:
    print("❌ No GPU — go to Runtime → Change runtime type → GPU")

## 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = "/content/drive/MyDrive/jit_rcdm"
os.makedirs(DRIVE_DIR, exist_ok=True)

print(f"Drive mounted. Project folder: {DRIVE_DIR}")
print("Contents:")
for f in sorted(os.listdir(DRIVE_DIR)):
    full = os.path.join(DRIVE_DIR, f)
    tag  = f"({os.path.getsize(full)/1e6:.1f} MB)" if os.path.isfile(full) else "(folder)"
    print(f"  {f} {tag}")

## 3 — Clone / pull JiT-RCDM code

> **After a local `git push`:** re-run this cell to pull the latest changes, then re-run Cell 7 to reload imports.

In [ ]:
import os, sys

REPO_DIR = "/content/jit_rcdm"
REPO_URL = "https://github.com/SeverinLe/master_implementation.git"
BRANCH   = "claude/silly-faraday-d8512b"

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print("Repo already cloned — pulling latest changes...")
    !git -C {REPO_DIR} pull
else:
    print(f"Cloning branch {BRANCH} ...")
    !git clone --branch {BRANCH} --single-branch {REPO_URL} {REPO_DIR}

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

%cd {REPO_DIR}

# Verify JiT code is present
assert os.path.exists("rcdm/jit.py"),       "❌ rcdm/jit.py missing — git pull may have failed"
assert os.path.exists("scripts/train.py"),  "❌ scripts/train.py missing"
print("\n✓ JiT-RCDM code present")
!ls rcdm/

## 4 — Install dependencies

In [ ]:
!pip install -q transformers safetensors wandb
print("✓ Dependencies installed")

from transformers import AutoModel
print("✓ transformers import OK (PyTorch >= 2.4 confirmed)")

## 5 — Log in to Weights & Biases

Get your API key from [wandb.ai/authorize](https://wandb.ai/authorize)

In [ ]:
import wandb
wandb.login()  # prompts for API key — paste from wandb.ai/authorize
print("✓ W&B logged in")
print(f"  Project URL will be: https://wandb.ai/{wandb.api.default_entity}/jit-rcdm")

## 6 — Copy DinoV3 checkpoint from Drive

In [ ]:
import shutil, os, json

SRC = "/content/drive/MyDrive/jit_rcdm/dinov3_vits16_tmp"
DST = "/content/jit_rcdm/checkpoints/dinov3_vits16_tmp"

assert os.path.isdir(SRC), (
    f"DinoV3 checkpoint not found at {SRC}.\n"
    "Upload dinov3_vits16_tmp/ to Google Drive at MyDrive/jit_rcdm/"
)

if not os.path.isdir(DST):
    shutil.copytree(SRC, DST)
    print(f"✓ Copied DinoV3 checkpoint → {DST}")
else:
    print(f"✓ DinoV3 checkpoint already at {DST}")

cfg = json.load(open(os.path.join(DST, "config.json")))
assert cfg['use_gated_mlp'] == False, "use_gated_mlp must be false — check config.json"
assert cfg['hidden_size']   == 384,   "Expected hidden_size=384"
print(f"  hidden_size={cfg['hidden_size']}, use_gated_mlp={cfg['use_gated_mlp']}, "
      f"num_register_tokens={cfg['num_register_tokens']}")
print("✓ Config OK")

## 7 — Copy training representations from Drive

In [ ]:
import shutil, os, torch

SRC = "/content/drive/MyDrive/jit_rcdm/train_reps.pt"
DST = "/content/jit_rcdm/data/messidor2/train_reps.pt"

assert os.path.exists(SRC), (
    f"train_reps.pt not found at {SRC}.\n"
    "Run precompute_reps.py locally first, then upload to Google Drive."
)

os.makedirs(os.path.dirname(DST), exist_ok=True)
if not os.path.exists(DST):
    print("Copying train_reps.pt from Drive...")
    shutil.copy2(SRC, DST)
else:
    print("train_reps.pt already present")

data = torch.load(DST, map_location="cpu")
reps = data["reps"]
assert reps.shape[1] == 384, f"Expected 384-dim reps, got {reps.shape[1]}"
print(f"✓ {len(data['paths'])} images, reps {tuple(reps.shape)}, "
      f"mean norm={reps.norm(dim=1).mean():.2f}")

## 8 — Verify full pipeline

> Re-run this cell after a `git pull` to reload updated code.

In [ ]:
# Reload modules cleanly after a git pull
import importlib, sys
for mod in list(sys.modules.keys()):
    if mod.startswith("rcdm"):
        del sys.modules[mod]

import sys, os
sys.path.insert(0, "/content/jit_rcdm")

from rcdm.encoder      import load_encoder, build_transform, DINOV3_CHECKPOINT
from rcdm.jit          import JiT_S_16, FlowMatching, create_jit_model
from rcdm.dataset      import RepresentationDataset
from rcdm.conditioning import RMSNorm, AdaLNZero, ConditioningProjector
import torch

print("✓ All imports OK")
print(f"  DINOV3_CHECKPOINT → {DINOV3_CHECKPOINT}")
assert os.path.isdir(DINOV3_CHECKPOINT), f"Checkpoint not found at {DINOV3_CHECKPOINT}"
print("  Checkpoint directory exists ✓")

# Forward-pass smoke test (CPU)
m = JiT_S_16(image_size=224, h_dim=384)
m.eval()
with torch.no_grad():
    out = m(torch.randn(2,3,224,224), torch.rand(2), torch.randn(2,384))
assert out.shape == (2,3,224,224)
n = sum(p.numel() for p in m.parameters())
print(f"✓ JiT_S_16 forward pass — output {tuple(out.shape)}, {n/1e6:.1f}M params")
print(f"  null_h  : shape={tuple(m.null_h.shape)}, requires_grad={m.null_h.requires_grad}")
print(f"  freqs_cis: shape={tuple(m.freqs_cis.shape)}, dtype={m.freqs_cis.dtype}")
print("\n✓ Pipeline ready — proceed to training")

## 9 — Train

Checkpoints saved to Drive every 5 000 steps — survive session disconnects.

In [ ]:
import torch
mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
batch  = 256 if mem_gb >= 70 else (128 if mem_gb >= 38 else 32)
print(f"GPU VRAM: {mem_gb:.0f} GB → batch_size={batch}")

In [ ]:
import os
CKPT_DIR = "/content/drive/MyDrive/jit_rcdm/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

!python scripts/train.py \
    --model          S16 \
    --reps_file      data/messidor2/train_reps.pt \
    --save_dir       {CKPT_DIR} \
    --image_size     224 \
    --batch_size     {batch} \
    --grad_accum     1 \
    --lr             1e-4 \
    --warmup_steps   2000 \
    --total_steps    100000 \
    --save_interval  5000 \
    --log_interval   100 \
    --cfg_dropout    0.1 \
    --ema_decay      0.9999 \
    --wandb_project  jit-rcdm \
    --wandb_run_name S16-100k-A100 \
    --device         cuda

## 9b — Resume after disconnection

Re-run cells 1–8, then run this cell. Latest checkpoint is picked up automatically.

In [ ]:
import os, glob, torch
CKPT_DIR = "/content/drive/MyDrive/jit_rcdm/checkpoints"
mem_gb   = torch.cuda.get_device_properties(0).total_memory / 1e9
batch    = 256 if mem_gb >= 70 else (128 if mem_gb >= 38 else 32)

ckpts  = sorted(glob.glob(os.path.join(CKPT_DIR, "jit_rcdm_step*.pt")))
assert ckpts, f"No checkpoint found in {CKPT_DIR}"
latest = ckpts[-1]
print(f"Resuming from: {latest}")

!python scripts/train.py \
    --model          S16 \
    --reps_file      data/messidor2/train_reps.pt \
    --save_dir       {CKPT_DIR} \
    --image_size     224 \
    --batch_size     {batch} \
    --grad_accum     1 \
    --lr             1e-4 \
    --warmup_steps   2000 \
    --total_steps    100000 \
    --save_interval  5000 \
    --log_interval   100 \
    --cfg_dropout    0.1 \
    --ema_decay      0.9999 \
    --resume         {latest} \
    --wandb_project  jit-rcdm \
    --wandb_run_name S16-100k-A100 \
    --device         cuda

## 10 — Sample

| Steps trained | `--cfg_scale` |
|---|---|
| < 15 k | 1.0 |
| 15–30 k | 1.5 |
| 30–50 k | 2.0 |
| > 50 k | 2.0–3.0 |

In [ ]:
import os, glob
CKPT_DIR   = "/content/drive/MyDrive/jit_rcdm/checkpoints"
SAMPLE_DIR = "/content/drive/MyDrive/jit_rcdm/samples"
os.makedirs(SAMPLE_DIR, exist_ok=True)

ckpts  = sorted(glob.glob(os.path.join(CKPT_DIR, "jit_rcdm_step*.pt")))
latest = ckpts[-1] if ckpts else os.path.join(CKPT_DIR, "jit_rcdm_final.pt")
print(f"Sampling from: {latest}")

TEST_IMAGES = sorted(glob.glob("/content/drive/MyDrive/jit_rcdm/test_images/*.png"))
assert TEST_IMAGES, "Upload .png test images to Drive at MyDrive/jit_rcdm/test_images/"
cond_args = " ".join(TEST_IMAGES)
print(f"Conditioning images: {len(TEST_IMAGES)}")

!python scripts/sampling.py \
    --checkpoint     {latest} \
    --cond_images    {cond_args} \
    --out_dir        {SAMPLE_DIR} \
    --n_samples      4 \
    --num_steps      50 \
    --cfg_scale      1.0 \
    --wandb_project  jit-rcdm \
    --wandb_run_name S16-samples \
    --device         cuda

## 11 — Monitor

In [ ]:
!nvidia-smi
!ls -lh /content/drive/MyDrive/jit_rcdm/checkpoints/ 2>/dev/null || echo "No checkpoints yet"

---

## Drive layout

```
MyDrive/jit_rcdm/
    dinov3_vits16_tmp/     ← upload before running
        config.json
        model.safetensors
    train_reps.pt          ← upload before running
    test_images/           ← upload before sampling
    checkpoints/           ← auto-created during training
    samples/               ← auto-created during sampling
```